# Lesson 10 - AI Agents for Production

For dis lesson, you go learn **production patterns** for AI agents wey dey use Microsoft Agent Framework (Python). We go cover:

- **Observability** — to dey add timing and logging for how agent dey interact
- **Evaluation** — to use evaluator agent to take score how good response be
- **Cost management** — ways to optimize token and how to pick better model

The koko na **travel agent** wey dey help users plan trips, plus monitoring and evaluation to dey ona top.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ting Dem Wey You Gats Think About for Production

To commot AI agents from prototype go production, you gats put eye well well for these three main tins:

1. **Observability** — You need see wetin di agent dey do, how long e dey take, and which kain tools e dey use. If you no dey do tracing and logging, e go hard well to find wahala for production.

2. **Evaluation** — Automated quality check go make sure say di agent answers dey correct, full, and helpful as time dey go. One evaluator agent fit take score the answers based on how dem set the standard.

3. **Cost Management** — How you dey use tokens go affect how much e go cost. Strategy wey like prompt optimization, model selection, and caching fit help hold down the cost without make quality drop.


## How to Build an Observable Agent

We de define travel tools and we wrap di agent call wit timing so we fit monitor latency. For production, you go fit connect am wit OpenTelemetry or similar tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

Di common production way na to use second agent as **evaluator**. Di evaluator dey score di main agent response based on set criteria like how complete e be, correctnes, and how e fit help.

Dis one fit:
- Automatic quality check before response reach users
- Find regression wen prompts or models change
- Keep eye dey on agent work time to time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Cost Management Strategies

Kontrol cost na important for production AI agents dem. Na dia main strategies be dis:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Make system instructions short and clear. Remove any repeat tins to reduce input tokens. |
    "| **Model selection** | Use smaller, cheap models (like GPT-4.1-mini) for simple tasks like classification or extraction, and keep bigger models for heavy reasoning. |\n",
| **Caching** | Keep tool results and popular questions ready to avoid repeat API calls. |
| **Token budgets** | Set `max_tokens` limits so the responses no go too long unexpectedly. |
| **Batching** | Put many user questions together inside one API call if e fit. |

For practice, one-level system dey work well: send simple requests to quick, cheap model and only pass big, complex questions go better model.


## Summary

For dis lesson, you learn how to:

1. **Add observability** to agent interactions with timing and logging, lay di foundation for tracing and monitoring.
2. **Evaluate agent responses** automatically using evaluator agent weh dey score completeness, accuracy, and helpfulness.
3. **Manage costs** through prompt optimization, model selection, caching, and token budgets.

Dis production patterns dey help make sure say your AI agents dey reliable, measurable, and cost-effective for scale.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
